# Actividad 2A. Cuaderno Jupyter: Preprocesamiento de imágenes con OpenCV
**Procesamiento Digital de Imágenes · MCDI · INFOTEC · 2027-1** · Unidad 2

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egonzalez1989/pdi-mcdi/blob/main/plantillas/2A_Plantilla_Preprocesamiento.ipynb)

| | |
|---|---|
| **Estudiante** | *Escribe aquí tu nombre completo* |
| **Tipo de actividad** | Individual |
| **Fecha límite** | Domingo 14 de febrero de 2027, 23:59 hrs. |
| **Puntaje** | 15 puntos |

**Propósito.** Aplicar y comparar técnicas de preprocesamiento, justificando la elección con criterios visuales y métricas cuantitativas.

**Producto esperado.** Una libreta Jupyter bien documentada que incluya:
- Transformaciones de intensidad (ajuste lineal y una transformación no lineal, por ejemplo, gamma).
- Análisis de histogramas antes y después de la ecualización (global y CLAHE).
- Filtros espaciales sobre imágenes con ruido, comparados con PSNR y SSIM.
- Operaciones morfológicas tras una binarización y transformaciones geométricas.
- Comentarios breves en Markdown interpretando los resultados.

### Instrucciones
1. Descarga el cuaderno plantilla 2A_Plantilla_Preprocesamiento.ipynb y selecciona al menos una imagen de tu interés (puede ser distinta a la de la Unidad 1).
2. Transformaciones de intensidad y contraste: compara la imagen original con un ajuste lineal de contraste y una corrección gamma; comenta en qué contextos sería útil cada transformación.
3. Histogramas y ecualización: calcula el histograma de la imagen original y de la transformada; aplica ecualización global y CLAHE y comenta en qué regiones mejora (o no) el contraste.
4. Filtrado espacial y métricas: añade ruido gaussiano y ruido sal y pimienta a una copia de la imagen; aplica al menos dos filtros (promedio, gaussiano, mediana) y construye una tabla con PSNR y SSIM respecto a la imagen original para elegir el filtro más adecuado a cada tipo de ruido. Opcional: aplica un filtro pasa-bajas en el dominio de la frecuencia (Fourier) y compáralo con el gaussiano.
5. Operaciones morfológicas: binariza la imagen (por ejemplo, con Otsu) y aplica erosión, dilatación y al menos una combinación (apertura o cierre); describe su efecto sobre los objetos y el ruido.
6. Transformaciones geométricas: aplica una rotación y un cambio de tamaño y comenta en qué problemas sería importante alinear la imagen o normalizar su tamaño antes de extraer características.
7. Estructura tu libreta con secciones y explica en Markdown qué operación aplicas, con qué parámetros y qué observas; justifica tus decisiones de preprocesamiento con las métricas obtenidas.
8. Guarda tu libreta como 2A_Nombre_Apellido.ipynb, expórtala a PDF con el mismo nombre y entrega ambos archivos.

### Lista de cotejo (así se evaluará tu trabajo)
| Criterio | Puntaje |
|---|---|
| Transformaciones de intensidad y análisis de histogramas (incluye ecualización global y CLAHE). | 4 |
| Aplicación y comparación de filtros espaciales / reducción de ruido, con tabla de PSNR y SSIM. | 4 |
| Uso adecuado de operaciones morfológicas y transformaciones geométricas. | 3 |
| Claridad en el código, visualizaciones y documentación en Markdown. | 4 |
| **Total** | **15** |

> 💡 Escribe tus interpretaciones en celdas Markdown como esta. Una figura sin interpretación no demuestra aprendizaje. Si usas fragmentos de código de otras fuentes (incluidos asistentes de IA), cítalos en la sección final.

In [ ]:
# Configuración del entorno. En Google Colab todas estas bibliotecas ya vienen instaladas.
import sys, platform
import numpy as np
import matplotlib.pyplot as plt
import cv2
import skimage
from skimage import data

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 10, "image.cmap": "gray"})
np.random.seed(42)

def mostrar(imagenes, titulos=None, cols=None, cmap="gray", tam=3.2, ejes=False):
    """Muestra una lista de imágenes en una cuadrícula. Las imágenes a color deben estar en RGB."""
    if not isinstance(imagenes, (list, tuple)):
        imagenes = [imagenes]
    n = len(imagenes)
    cols = cols or min(n, 4)
    filas = int(np.ceil(n / cols))
    fig, axs = plt.subplots(filas, cols, figsize=(tam * cols, tam * filas), squeeze=False)
    for i, ax in enumerate(axs.flat):
        if i < n:
            im = imagenes[i]
            ax.imshow(im, cmap=None if im.ndim == 3 else cmap)
            if titulos:
                ax.set_title(titulos[i])
        if not ejes or i >= n:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"Python {platform.python_version()} | NumPy {np.__version__} | OpenCV {cv2.__version__} | scikit-image {skimage.__version__}")

# Datos del curso: se buscan en la carpeta datos/ del repositorio (si lo clonaste) o se descargan.
import os, io, zipfile, urllib.request
def carpeta_datos(url="https://github.com/egonzalez1989/pdi-mcdi/archive/refs/heads/main.zip"):
    for d in ("datos", "../datos", "pdi-mcdi-main/datos"):
        if os.path.isdir(d):
            return d
    try:
        with urllib.request.urlopen(url, timeout=20) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
        return "pdi-mcdi-main/datos"
    except Exception as e:
        print("No se pudieron descargar los datos del curso:", e)
        return None
DATOS = carpeta_datos()
print("Carpeta de datos:", DATOS)

import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim
from skimage.util import random_noise, img_as_ubyte

## 0. Imagen de trabajo
Elige una imagen con un problema de calidad (bajo contraste, ruido o iluminación desigual). Por omisión se usa `datos/imagenes/luna.png`.

In [ ]:
RUTA = ""
if not RUTA and DATOS:
    RUTA = os.path.join(DATOS, "imagenes", "luna.png")
gris = cv2.imread(RUTA, cv2.IMREAD_GRAYSCALE) if RUTA else None
if gris is None:
    gris = data.moon()
mostrar(gris, "Imagen original")

## 1. Transformaciones de intensidad y contraste

In [ ]:
# TODO: ajuste lineal de contraste (cv2.convertScaleAbs con alpha, beta) y corrección gamma (con una LUT)

*¿En qué contextos sería útil cada transformación?* ...

## 2. Histogramas y ecualización (global y CLAHE)

In [ ]:
# TODO: histogramas antes/después; cv2.equalizeHist y cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

*¿Mejora el contraste percibido? ¿En qué regiones?* ...

## 3. Ruido, filtros espaciales y métricas (PSNR / SSIM)

In [ ]:
ruidos = {
    "gaussiano": img_as_ubyte(random_noise(gris, mode="gaussian", var=0.01, rng=1)),
    "sal_pimienta": img_as_ubyte(random_noise(gris, mode="s&p", amount=0.05, rng=1)),
}
filtros = {
    "promedio_5": lambda x: cv2.blur(x, (5, 5)),
    # TODO: agrega al menos "gaussiano_5" (cv2.GaussianBlur) y "mediana_5" (cv2.medianBlur)
}
filas = []
for nr, r in ruidos.items():
    filas.append((nr, "sin filtro", psnr(gris, r), ssim(gris, r)))
    for nf, f in filtros.items():
        out = f(r)
        filas.append((nr, nf, psnr(gris, out), ssim(gris, out)))
tabla = pd.DataFrame(filas, columns=["ruido", "filtro", "PSNR (dB)", "SSIM"]).round(3)
tabla

*¿Qué filtro elegirías para cada tipo de ruido? Justifica con la tabla:* ...

### (Opcional) Filtrado en el dominio de la frecuencia

In [ ]:
# TODO opcional: F = np.fft.fftshift(np.fft.fft2(gris)); muestra np.log1p(np.abs(F));
#       aplica un filtro gaussiano pasa-bajas en frecuencia y compáralo con cv2.GaussianBlur

## 4. Operaciones morfológicas

In [ ]:
# TODO: binariza con Otsu (cv2.threshold(..., cv2.THRESH_BINARY + cv2.THRESH_OTSU))
# TODO: aplica erosión, dilatación y apertura o cierre (cv2.morphologyEx) con un elemento estructurante

*Describe el efecto de cada operación sobre los objetos y el ruido:* ...

## 5. Transformaciones geométricas

In [ ]:
# TODO: aplica una rotación y un cambio de tamaño

*¿En qué problemas sería importante alinear o normalizar el tamaño antes de extraer características?* ...

## 6. Conclusión: ¿qué esquema de preprocesamiento recomiendas y por qué?
...

## Referencias y uso de herramientas
*Lista en formato APA las fuentes consultadas. Si usaste asistentes de IA generativa, indica cuál, para qué y verifica que entiendes cada línea de código que entregas.*

---
**Antes de entregar:** `Entorno de ejecución → Reiniciar y ejecutar todo`, verifica que no haya errores y exporta a PDF (`Archivo → Imprimir → Guardar como PDF` o `jupyter nbconvert --to webpdf`).